In [1]:
library(SingleCellExperiment)
library(dplyr)
library(data.table)
library(purrr)

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOrderStats, rowProds, rowQuantiles, rowRanges

In [2]:
io = list()
io$atlas_sce = "/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/embryo_sce.rds"

In [3]:
extended.sce = readRDS(io$atlas_sce)
meta_extended = colData(extended.sce) %>% 
    as.data.table() %>% 
    .[embryo_version=='Original']
rm(extended.sce)

In [5]:
meta_extended_celltypes = meta_extended[,c('cell', 'celltype_extended_atlas')] %>% 
    setnames('celltype_extended_atlas', 'celltype_extended') %>% 
    .[,celltype_extended := gsub(' ', '_', gsub('/', '_', celltype_extended))]  # change spaces & slash to "_"

In [6]:
tail(meta_extended_celltypes)

cell,celltype_extended
<fct>,<chr>
cell_139325,Somitic_mesoderm
cell_139326,Erythroid
cell_139327,Erythroid
cell_139329,Somitic_mesoderm
cell_139330,Erythroid
cell_139331,Optic_vesicle


In [7]:
meta_original = fread('sample_metadata.txt.gz') %>% 
    .[,index:=as.numeric(stringr::str_split(cell, '_') %>% map_chr(2))] %>% 
    .[order(index)]

In [8]:
summary(meta_original$cell == meta_extended_celltypes$cell)

   Mode    TRUE 
logical  116312 

In [9]:
meta_complete = merge(meta_original, meta_extended_celltypes, by='cell') %>%
    .[order(index)]

In [10]:
fwrite(meta_complete, 'sample_metadata_extended.txt.gz')